System information (for reproducibility):

In [11]:
versioninfo()

Julia Version 1.12.6
Commit 15346901f00 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 12 × Apple M2 Max
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m2)
  GC: Built with stock GC
Threads: 8 default, 1 interactive, 8 GC (on 8 virtual cores)
Environment:
  JULIA_NUM_THREADS = 8
  JULIA_EDITOR = code


Load packages:

In [12]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Documents/github.com/ucla-biostat-257/2026spring/slides/15-linreg`


Status `~/Documents/github.com/ucla-biostat-257/2026spring/slides/15-linreg/Project.toml`
  [6e4b80f9] BenchmarkTools v1.8.0
  [7522ee7d] SweepOperator v0.3.4
  [37e2e46d] LinearAlgebra v1.12.0


## Comparing methods for linear regression

Methods for solving linear regression $\widehat \beta = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$:

| Method            | Flops                  | Remarks                 | Software | Stability   |
| :---------------: | :--------------------: | :---------------------: | :------: | :---------: |
| Sweep             | $np^2 + p^3$           | $(X^TX)^{-1}$ available | SAS      | less stable |
| Cholesky          | $np^2 + p^3/3$         |                         |          | less stable |
| QR by Householder | $2np^2 - (2/3)p^3$     |                         | R        | stable      |
| QR by MGS         | $2np^2$                | $Q_1$ available         |          | stable      | 
| QR by SVD         | $4n^2p + 8np^2 + 9p^3$ | $X = UDV^T$             |          | most stable |  

Remarks:

1. When $n \gg p$, sweep and Cholesky are twice faster than QR and need less space.  
2. Sweep and Cholesky are based on the **Gram matrix** $\mathbf{X}^T \mathbf{X}$, which can be dynamically updated with incoming data. They can handle huge $n$, moderate $p$ data sets that cannot fit into memory.  
3. QR methods are more stable and produce numerically more accurate solution.  
4. Although sweep is slower than Cholesky, it yields standard errors and so on.  
5. MGS appears slower than Householder, but it yields $\mathbf{Q}_1$.

> **There is simply no such thing as a universal 'gold standard' when it comes to algorithms.**

## Benchmark

In [13]:
using SweepOperator, BenchmarkTools, LinearAlgebra

linreg_cholesky(y::Vector, X::Matrix) = cholesky!(X'X) \ (X'y)

linreg_qr(y::Vector, X::Matrix) = X \ y

function linreg_sweep(y::Vector, X::Matrix)
    p = size(X, 2)
    xy = [X y]
    tableau = xy'xy
    sweep!(tableau, 1:p)
    return tableau[1:p, end]
end

function linreg_svd(y::Vector, X::Matrix)
    xsvd = svd(X)
    return xsvd.V * ((xsvd.U'y) ./ xsvd.S)
end

linreg_svd (generic function with 1 method)

In [14]:
using Random

Random.seed!(123) # seed

n, p = 10, 3
X = randn(n, p)
y = randn(n)

# check these methods give same answer
@show linreg_cholesky(y, X)
@show linreg_qr(y, X)
@show linreg_sweep(y, X)
@show linreg_svd(y, X);

linreg_cholesky(y, X) = [-0.07196570434574734, -0.1357569945585938, -0.1882019968945651]
linreg_qr(y, X) = [-0.07196570434574737, -0.13575699455859397, -0.18820199689456493]
linreg_sweep(y, X) = [-0.07196570434574734, -0.1357569945585939, -0.188201996894565]
linreg_svd(y, X) = [-0.07196570434574741, -0.1357569945585938, -0.18820199689456527]


In [15]:
n, p = 1000, 300
X = randn(n, p)
y = randn(n)

@benchmark linreg_cholesky(y, X)

BenchmarkTools.Trial: 6476 samples with 1 evaluation per sample.
 Range (min … max):  710.459 μs …   5.326 ms  ┊ GC (min … max): 0.00% … 85.13%
 Time  (median):     744.792 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   771.906 μs ± 124.912 μs  ┊ GC (mean ± σ):  2.13% ±  6.68%

  ▃▄▆██▇▅▄▃▂▁                                                   ▁
  ████████████▇▇▇▇█▇█▇▇▇▆▇▇▇▇▆▅▅▅▅▄▄▂▃▄▄▃▃▂▅▃▄▆▇▅▇▇▇▇▇▇▆▆▆▆▄▅▅▅ █
  710 μs        Histogram: log(frequency) by time       1.19 ms <

 Memory estimate: 709.20 KiB, allocs estimate: 9.

In [16]:
@benchmark linreg_sweep(y, X)

BenchmarkTools.Trial: 1072 samples with 1 evaluation per sample.
 Range (min … max):  4.436 ms … 48.614 ms  ┊ GC (min … max): 0.00% … 90.67%
 Time  (median):     4.509 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.666 ms ±  1.468 ms  ┊ GC (mean ± σ):  2.48% ±  4.51%

   ▅█▃▃▁                                                      
  ▃█████▆▅▅▅▄▄▄▃▄▃▃▃▂▃▃▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▃▁▂▂▂▂ ▃
  4.44 ms        Histogram: frequency by time        5.18 ms <

 Memory estimate: 3.01 MiB, allocs estimate: 12.

In [17]:
@benchmark linreg_qr(y, X)

BenchmarkTools.Trial: 391 samples with 1 evaluation per sample.
 Range (min … max):  12.340 ms …  22.049 ms  ┊ GC (min … max): 0.00% … 42.13%
 Time  (median):     12.682 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   12.805 ms ± 584.687 μs  ┊ GC (mean ± σ):  0.76% ±  2.61%

      ▂▄▇█▇▆▃                                                   
  ▃▃▃▅████████▇▆▅▆▃▆▅▆▅▅▃▄▃▄▃▃▂▁▁▃▁▂▂▁▁▂▃▁▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂ ▃
  12.3 ms         Histogram: frequency by time         14.5 ms <

 Memory estimate: 2.44 MiB, allocs estimate: 28.

In [18]:
@benchmark linreg_svd(y, X)

BenchmarkTools.Trial: 197 samples with 1 evaluation per sample.
 Range (min … max):  23.306 ms … 62.522 ms  ┊ GC (min … max): 0.00% … 60.88%
 Time  (median):     24.422 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   25.497 ms ±  4.793 ms  ┊ GC (mean ± σ):  3.15% ±  8.04%

  ▄█▅▂                                                         
  ██████▅▆█▄▄▄▄▁▄▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄ ▄
  23.3 ms      Histogram: log(frequency) by time      61.1 ms <

 Memory estimate: 8.08 MiB, allocs estimate: 27.